# YAMNet Exploration

Goal:
Understand how pretrained audio models interpret sound.

Questions:

- What does YAMNet output?
- What are embeddings?
- How fast is inference?
- Does it recognize our sounds?

In [127]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from pathlib import Path
import librosa
import pandas as pd

In [128]:
model = hub.load(
    "https://tfhub.dev/google/yamnet/1"
)

model

<tensorflow.python.saved_model.load.Loader._recreate_base_user_object.<locals>._UserObject at 0x2ceee7c7860>

In [129]:
ROOT = Path("../data/raw/esc50")

AUDIO = ROOT / "audio"

sample = AUDIO / "1-22694-A-20.wav"

In [130]:
waveform, sr = librosa.load(
    sample,
    sr=16000
)

waveform = waveform.astype(
    np.float32
)

print(sr)
print(waveform.shape)

16000
(80000,)


In [131]:
scores, embeddings, spectrogram = model(
    waveform
)

In [132]:
print(scores.shape)

print(embeddings.shape)

print(spectrogram.shape)

(10, 521)
(10, 1024)
(528, 64)


In [133]:
scores.numpy()[:3]

array([[3.1292029e-02, 4.8936099e-02, 4.9655419e-04, ..., 1.5239505e-05,
        1.5998743e-06, 1.1941684e-12],
       [1.0403191e-01, 9.6911982e-02, 8.4452686e-04, ..., 1.0601186e-04,
        1.3240791e-05, 4.4895612e-11],
       [1.2089270e-01, 2.7948424e-01, 1.3727183e-03, ..., 1.1700538e-06,
        2.2165456e-07, 2.8186212e-14]], shape=(3, 521), dtype=float32)

In [134]:
embeddings.numpy()[:2]

array([[0.        , 0.03677348, 0.10761797, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.01672826, ..., 0.        , 0.        ,
        0.        ]], shape=(2, 1024), dtype=float32)

Observation

- Scores: The model broke our 5-second audio clip down into 10 distinct temporal windows (frames). For each window, it outputs a vector of 521 probability scores, representing YAMNet's confidence across its 521 supported sound classes.
- Embeddings: The timeline matches the 10 windows seen in the scores. Each frame is represented by a highly dense 1,024-dimensional feature vector. These embeddings capture the core acoustic signature of the sound and are ideal for downstream transfer learning.
- Spectrogram: This represents YAMNet's internal preprocessing step. It computed a Log-Mel Spectrogram across the full audio clip containing 528 time slices and compressed the frequency information into 64 mel bins.

In [135]:
class_map = pd.read_csv(
    "https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv"
)

In [136]:
prediction = scores.numpy().mean(
    axis=0
)

In [137]:
top = prediction.argsort()[-10:][::-1]

class_map.iloc[top]

,index,mid,display_name
19,19,/m/0463cq4,"Crying, sobbing"
20,20,/t/dd00002,"Baby cry, infant cry"
4,4,/m/0261r1,Babbling
13,13,/m/01j3sz,Laughter
500,500,/t/dd00125,"Inside, small room"
54,54,/m/02p3nc,Hiccup
21,21,/m/07qz6j3,Whimper
17,17,/m/07sq110,Belly laugh
15,15,/m/07r660_,Giggle
1,1,/m/0ytgt,"Child speech, kid speaking"


Alarm:
strong

Knock:
good

Baby:
weak

In [138]:
import time

start = time.time()

model(
    waveform
)

end = time.time()

print(
    end-start
)

0.0775153636932373
